## Introducción a MCP y Herramientas de Agentes

**MCP (Multi-Agent Collaboration Platform)** es una plataforma que permite la colaboración y orquestación de múltiples agentes de IA para trabajar juntos en tareas complejas. Facilita la definición, el registro y la ejecución de herramientas que los agentes pueden utilizar para interactuar con el mundo exterior o realizar cálculos específicos.

Esta notebook explora el uso de MCP, una potente plataforma que facilita la colaboración entre diferentes agentes de IA. Uno de los aspectos fundamentales de MCP es la capacidad de definir y utilizar **herramientas de agentes**.

Las herramientas son funciones específicas que los agentes de IA pueden invocar para realizar tareas o interactuar con el entorno. En esta notebook, veremos ejemplos de cómo definir estas herramientas (como `calculate_average`, `calculate_median` y `calculate_standard_deviation`) y cómo un modelo de lenguaje puede utilizarlas para responder preguntas complejas que requieren cálculos o acceso a funcionalidades externas. Esto permite a los agentes ir más allá de la generación de texto, realizando acciones y obteniendo información de manera estructurada.

Tooling (o 'herramientas') se refiere al conjunto de utilidades, frameworks, librerías, y plataformas que se utilizan para desarrollar, desplegar, gestionar y operar sistemas, especialmente en el contexto de la inteligencia artificial y el desarrollo de software.

En el ámbito de los agentes de IA, como los que estamos viendo en esta notebook con MCP, 'tooling' se refiere específicamente a:

Las funciones o métodos que un agente de IA puede invocar: Estas son las 'herramientas' que el agente utiliza para realizar acciones, obtener información del mundo exterior (por ejemplo, acceder a APIs, bases de datos, sistemas de archivos) o realizar cálculos específicos (como calculate_average, calculate_median, etc.). Son las capacidades extendidas del agente.
Los frameworks y librerías que facilitan la creación y gestión de estas herramientas: Plataformas como Langchain o MCP (mcp.server) son ejemplos de 'tooling' que proveen la infraestructura para definir, registrar y permitir que los modelos de lenguaje interactúen con estas funciones de manera estructurada y segura.
El ecosistema de desarrollo completo: Incluye también editores de código, sistemas de control de versiones, entornos de integración continua/entrega continua (CI/CD), herramientas de monitoreo y depuración, etc., que son fundamentales para el ciclo de vida del desarrollo de sistemas basados en IA y agentes.
En resumen, el 'tooling' es todo lo que ayuda a los desarrolladores y a los propios agentes de IA a ser más eficientes y a expandir sus capacidades.

### 1. Instalación de Librerías Necesarias

### 2. Importación de Módulos

In [ ]:
!pip install -q -U langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 21.5 MB/s eta 0:00:00


In [ ]:
import os
import json
from langchain_openai import ChatOpenAI
from openai import OpenAI
from langchain.tools import tool


### 3. Configuración de Clave API

In [ ]:
from google.colab import userdata
os.environ["OPENROUTER_KEY"] = userdata.get('OPENROUTER_KEY')
OPENROUTER_API_KEY = os.getenv("OPENROUTER_KEY")

### 4. Inicialización del Cliente OpenAI y Definición del Modelo

In [ ]:
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

MODEL_NAME = "cohere/north-mini-code:free"

### 5. Configuración del Modelo de Lenguaje (LLM) con Langchain

In [ ]:
llm = ChatOpenAI(
    model="cohere/north-mini-code:free",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

### 6. Definición de Herramienta: Calcular Promedio (Langchain)

In [ ]:
from langchain.tools import tool


@tool
def calculate_average(values: list[float]) -> float:
    """Calculate the average of a list of numbers."""

    return sum(values) / len(values)

### 7. Demostración de la Herramienta `calculate_average`

In [ ]:
calculate_average.invoke({
    "values": [10, 20, 30, 40, 50]
})

30.0

### 8. Vinculación de Herramientas al LLM

In [ ]:
llm_with_tools = llm.bind_tools(
    [calculate_average]
)

### 9. Invocación del LLM con Herramientas para una Pregunta

In [ ]:
response = llm_with_tools.invoke(
    "What is the average of 10, 20, 30, 40 and 50?"
)

response.tool_calls

[{'name': 'calculate_average',
  'args': {'values': [10.0, 20.0, 30.0, 40.0, 50.0]},
  'id': 'calculate_average_s99xdjnsvvx8',
  'type': 'tool_call'}]

### 10. Inspección de las Llamadas a Herramientas Generadas por el LLM

In [ ]:
!pip install -q -U "mcp[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 10.9 MB/s eta 0:00:00


### 13. Inicialización del Servidor MCP

In [ ]:
from mcp.server import MCPServer

mcp = MCPServer("Data Science Tools")

### 14. Definición de Herramienta MCP: Calcular Promedio

In [ ]:
@mcp.tool()
def calculate_average(values: list[float]) -> float:
    """Calculate the average of a list of numbers."""

    return sum(values) / len(values)

### 15. Definición de Herramienta MCP: Calcular Mediana

In [ ]:
@mcp.tool()
def calculate_median(values: list[float]) -> float:
    """Calculate the median of a list of numbers."""

    values = sorted(values)
    n = len(values)

    middle = n // 2

    if n % 2 == 0:
        return (values[middle - 1] + values[middle]) / 2

    return values[middle]

### 16. Definición de Herramienta MCP: Calcular Desviación Estándar

In [ ]:
import statistics


@mcp.tool()
def calculate_standard_deviation(
    values: list[float]
) -> float:
    """Calculate the sample standard deviation."""

    return statistics.stdev(values)

### 17. Definición de la Pregunta para el LLM

In [ ]:
question = """
Calculate the mean, median and standard deviation
of [10, 12, 15, 20, 25].
"""

In [ ]:
for tool_call in response.tool_calls:
    print(tool_call)

{'name': 'calculate_average', 'args': {'values': [10.0, 20.0, 30.0, 40.0, 50.0]}, 'id': 'calculate_average_s99xdjnsvvx8', 'type': 'tool_call'}


### 11. Ejecución de la Herramienta `calculate_average` a partir de la Llamada del LLM

In [ ]:
for tool_call in response.tool_calls:
    if tool_call["name"] == "calculate_average":
        result = calculate_average(**tool_call["args"])
        print(result)

30.0
